# 🏦 Loan Approval Prediction System
## AIML Summer Internship 2026 — MNNIT Allahabad

**Dataset:** Kaggle Loan Prediction Dataset  
**Objective:** Predict whether a loan application will be approved based on applicant information.

---

### 📋 ML Lifecycle Covered:
1. Problem Understanding
2. Data Collection
3. Data Preprocessing
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Model Building
7. Model Evaluation
8. Model Deployment (Streamlit App)

## ⚙️ Step 0: Install Dependencies

In [ ]:
# Install required packages
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib kaggle

import warnings
warnings.filterwarnings('ignore')
print('✅ All packages installed successfully!')

## 📦 Step 1: Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
import joblib
import os
import json
import sys

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print(f'✅ Libraries imported | Python {sys.version.split()[0]} | sklearn {__import__("sklearn").__version__}')

## 🎯 Step 2: Problem Understanding

**Business Problem:**  
Banks receive thousands of loan applications daily. Manually reviewing each application is time-consuming and error-prone. An ML model can automate the initial screening, reducing processing time and human bias.

**ML Task:** Binary Classification (Loan Approved: Y / N)  
**Target Variable:** `Loan_Status` (Y = Approved, N = Rejected)

**Features Available:**
| Feature | Description |
|---|---|
| Gender | Male / Female |
| Married | Yes / No |
| Dependents | Number of dependents |
| Education | Graduate / Not Graduate |
| Self_Employed | Yes / No |
| ApplicantIncome | Income of applicant |
| CoapplicantIncome | Income of co-applicant |
| LoanAmount | Loan amount (in thousands) |
| Loan_Amount_Term | Term of loan in months |
| Credit_History | Credit history meets guidelines (1 = Yes) |
| Property_Area | Urban / Semiurban / Rural |

## 📥 Step 3: Data Collection

In [ ]:
# Option A: Download from Kaggle (requires API key)
# Uncomment if you have kaggle.json configured:
# !kaggle competitions download -c loan-prediction-problem-dataset
# !unzip -o loan-prediction-problem-dataset.zip -d /content/Dataset/

# Option B: Load directly from URL (backup)
TRAIN_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/loan_prediction.csv'

# Create Dataset directory
os.makedirs('/content/Dataset', exist_ok=True)
os.makedirs('/content/Model', exist_ok=True)
os.makedirs('/content/Notebook', exist_ok=True)

try:
    df = pd.read_csv(TRAIN_URL)
    df.to_csv('/content/Dataset/loan_data.csv', index=False)
    print(f'✅ Dataset loaded from URL | Shape: {df.shape}')
except Exception as e:
    print(f'⚠️  URL failed: {e}')
    print('🔁 Using embedded fallback dataset...')
    # Embedded fallback — a compact but realistic version
    from io import StringIO
    FALLBACK_CSV = """Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
LP001002,Male,No,0,Graduate,No,5849,0.0,,,1.0,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,Y
LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y
LP001014,Male,No,3+,Graduate,No,3036,2504.0,158.0,360.0,0.0,Semiurban,N
LP001018,Male,Yes,2,Graduate,No,4006,1526.0,168.0,360.0,1.0,Urban,Y
LP001020,Male,Yes,0,Graduate,No,12841,10968.0,349.0,360.0,1.0,Semiurban,N"""
    df = pd.read_csv(StringIO(FALLBACK_CSV))
    # Expand to 614 rows for proper training
    df = pd.concat([df]*62, ignore_index=True)
    df['Loan_ID'] = [f'LP{100000+i}' for i in range(len(df))]
    df.to_csv('/content/Dataset/loan_data.csv', index=False)
    print(f'✅ Fallback dataset created | Shape: {df.shape}')

print(f'\n📊 Dataset Info:')
print(f'   Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print('=' * 60)
print('📋 DATASET OVERVIEW')
print('=' * 60)
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nTarget Distribution:\n{df["Loan_Status"].value_counts()}')
print(f'\nApproval Rate: {(df["Loan_Status"]=="Y").mean()*100:.1f}%')

In [ ]:
print(df.describe(include='all').T)

In [ ]:
# ── EDA Figure 1: Target Distribution & Categorical Features ──
fig = plt.figure(figsize=(18, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('📊 EDA — Categorical Features vs Loan Status', fontsize=16, fontweight='bold', y=1.01)

cat_cols = ['Loan_Status', 'Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Dependents', 'Credit_History']

for i, col in enumerate(cat_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    if col == 'Loan_Status':
        vc = df[col].value_counts()
        ax.pie(vc, labels=['Approved', 'Rejected'], autopct='%1.1f%%',
               colors=['#4CAF50', '#F44336'], startangle=90)
        ax.set_title('Target: Loan Status', fontweight='bold')
    elif col == 'Credit_History':
        temp = df.groupby(col)['Loan_Status'].apply(lambda x: (x=='Y').mean()).reset_index()
        temp.columns = [col, 'Approval Rate']
        sns.barplot(data=temp, x=col, y='Approval Rate', ax=ax, palette='Set2')
        ax.set_title(f'{col} vs Approval Rate', fontweight='bold')
        ax.set_ylim(0, 1)
    else:
        data_plot = df.groupby([col, 'Loan_Status']).size().reset_index(name='Count')
        sns.barplot(data=data_plot, x=col, y='Count', hue='Loan_Status',
                    palette={'Y':'#4CAF50','N':'#F44336'}, ax=ax)
        ax.set_title(f'{col} vs Loan Status', fontweight='bold')
        ax.tick_params(axis='x', rotation=20)
        ax.legend(title='Status', fontsize=8)

plt.savefig('/content/Notebook/eda_categorical.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ EDA Figure 1 saved')

In [ ]:
# ── EDA Figure 2: Numerical Features Distribution ──
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('📊 EDA — Numerical Features Distribution', fontsize=16, fontweight='bold')

for i, col in enumerate(num_cols):
    # Histogram
    ax1 = axes[0, i]
    df[col].dropna().hist(ax=ax1, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax1.set_title(f'{col}\nHistogram', fontsize=10)
    ax1.set_xlabel('')

    # Boxplot by target
    ax2 = axes[1, i]
    approved = df[df['Loan_Status']=='Y'][col].dropna()
    rejected = df[df['Loan_Status']=='N'][col].dropna()
    ax2.boxplot([approved, rejected], labels=['Approved', 'Rejected'],
                patch_artist=True,
                boxprops=dict(facecolor='lightgreen', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
    ax2.set_title(f'{col}\nvs Loan Status', fontsize=10)

plt.tight_layout()
plt.savefig('/content/Notebook/eda_numerical.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ EDA Figure 2 saved')

In [ ]:
# ── EDA Figure 3: Correlation Heatmap ──
df_numeric = df.copy()
for col in df_numeric.select_dtypes('object').columns:
    df_numeric[col] = LabelEncoder().fit_transform(df_numeric[col].astype(str))

plt.figure(figsize=(12, 9))
corr = df_numeric.drop('Loan_ID', axis=1, errors='ignore').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('📊 Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/content/Notebook/eda_correlation.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ EDA Figure 3 saved')

## 🔧 Step 5: Data Preprocessing & Feature Engineering

In [ ]:
print('=' * 60)
print('🔧 DATA PREPROCESSING')
print('=' * 60)

# Work on a copy
data = df.copy()

# Drop Loan_ID — not a feature
data.drop('Loan_ID', axis=1, inplace=True, errors='ignore')

print(f'Missing values before imputation:\n{data.isnull().sum()}\n')

# ── Categorical Imputation ──
cat_features = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in cat_features:
    if col in data.columns:
        mode_val = data[col].mode()[0]
        data[col].fillna(mode_val, inplace=True)
        print(f'  [{col}] → filled with mode = {mode_val}')

# ── Numerical Imputation ──
for col in ['LoanAmount']:
    median_val = data[col].median()
    data[col].fillna(median_val, inplace=True)
    print(f'  [{col}] → filled with median = {median_val:.1f}')

print(f'\nMissing values after imputation:\n{data.isnull().sum()}')

In [ ]:
# ── Feature Engineering ──
print('\n🧪 FEATURE ENGINEERING')
print('-' * 40)

# 1. Total Income
data['TotalIncome'] = data['ApplicantIncome'] + data['CoapplicantIncome']

# 2. Log transforms (reduce skewness)
data['Log_ApplicantIncome']   = np.log1p(data['ApplicantIncome'])
data['Log_CoapplicantIncome'] = np.log1p(data['CoapplicantIncome'])
data['Log_LoanAmount']        = np.log1p(data['LoanAmount'])
data['Log_TotalIncome']       = np.log1p(data['TotalIncome'])

# 3. EMI = LoanAmount / Loan_Amount_Term
data['EMI'] = data['LoanAmount'] / data['Loan_Amount_Term'].replace(0, np.nan)
data['EMI'].fillna(data['EMI'].median(), inplace=True)

# 4. Balance Income = TotalIncome - EMI
data['Balance_Income'] = data['TotalIncome'] - (data['EMI'] * 1000)

# 5. Debt to Income Ratio
data['Debt_Income_Ratio'] = data['LoanAmount'] / data['TotalIncome'].replace(0, np.nan)
data['Debt_Income_Ratio'].fillna(data['Debt_Income_Ratio'].median(), inplace=True)

print('  ✅ TotalIncome')
print('  ✅ Log_ApplicantIncome, Log_CoapplicantIncome, Log_LoanAmount, Log_TotalIncome')
print('  ✅ EMI (Estimated Monthly Installment)')
print('  ✅ Balance_Income')
print('  ✅ Debt_Income_Ratio')

# ── Drop raw columns after log transform ──
cols_to_drop = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'TotalIncome']
data.drop(columns=cols_to_drop, inplace=True)

print(f'\nShape after feature engineering: {data.shape}')
data.head(3)

In [ ]:
# ── Encode Categorical Variables ──
print('\n🏷️  LABEL ENCODING')
print('-' * 40)

# Fix Dependents — '3+' → 3
data['Dependents'] = data['Dependents'].astype(str).str.replace('+', '', regex=False).astype(float)

label_encoders = {}
cat_encode_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']

for col in cat_encode_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le
    print(f'  [{col}] → classes: {list(le.classes_)}')

# Encode target
data['Loan_Status'] = (data['Loan_Status'] == 'Y').astype(int)
print(f'\n  [Loan_Status] → Y=1, N=0')
print(f'\nTarget distribution:\n{data["Loan_Status"].value_counts()}')

# Save encoders
joblib.dump(label_encoders, '/content/Model/label_encoders.pkl')
print('\n✅ Label encoders saved to /content/Model/label_encoders.pkl')

In [ ]:
# ── Train-Test Split ──
print('\n✂️  TRAIN-TEST SPLIT')
print('-' * 40)

X = data.drop('Loan_Status', axis=1)
y = data['Loan_Status']

FEATURE_NAMES = list(X.columns)
print(f'Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]} samples')
print(f'Train class balance: {dict(y_train.value_counts())}')
print(f'Test  class balance: {dict(y_test.value_counts())}')

# Save feature names
joblib.dump(FEATURE_NAMES, '/content/Model/feature_names.pkl')
print('\n✅ Feature names saved')

## 🤖 Step 6: Model Building

In [ ]:
# ── Define Models ──
print('🤖 DEFINING MODELS')
print('=' * 60)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, '/content/Model/scaler.pkl')
print('✅ StandardScaler fitted and saved')

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, C=1.0, solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )
}

print(f'\n📌 Models to evaluate:')
for name in models:
    print(f'   - {name}')

In [ ]:
# ── Train & Evaluate All Models ──
results = {}
trained_models = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\n🚀 Training: {name}')
    print('-' * 50)

    # Use scaled data for LR, raw for tree-based
    if name == 'Logistic Regression':
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train.values, X_test.values

    # Fit
    model.fit(X_tr, y_train)
    trained_models[name] = model

    # Predictions
    y_pred      = model.predict(X_te)
    y_pred_prob = model.predict_proba(X_te)[:, 1]

    # Cross-validation
    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    # Metrics
    metrics = {
        'Accuracy'  : accuracy_score(y_test, y_pred),
        'Precision' : precision_score(y_test, y_pred, zero_division=0),
        'Recall'    : recall_score(y_test, y_pred, zero_division=0),
        'F1_Score'  : f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC'   : roc_auc_score(y_test, y_pred_prob),
        'CV_Mean'   : cv_scores.mean(),
        'CV_Std'    : cv_scores.std(),
    }
    results[name] = metrics

    print(f'  Accuracy  : {metrics["Accuracy"]*100:.2f}%')
    print(f'  Precision : {metrics["Precision"]*100:.2f}%')
    print(f'  Recall    : {metrics["Recall"]*100:.2f}%')
    print(f'  F1-Score  : {metrics["F1_Score"]*100:.2f}%')
    print(f'  ROC-AUC   : {metrics["ROC_AUC"]*100:.2f}%')
    print(f'  CV (5-fold): {metrics["CV_Mean"]*100:.2f}% ± {metrics["CV_Std"]*100:.2f}%')
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

## 📊 Step 7: Model Evaluation & Comparison

In [ ]:
# ── Comparison Table ──
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('ROC_AUC', ascending=False)
print('📊 MODEL COMPARISON TABLE')
print('=' * 70)
print(results_df.to_string(float_format='{:.4f}'.format))

best_model_name = results_df['ROC_AUC'].idxmax()
print(f'\n🏆 Best Model: {best_model_name} (ROC-AUC = {results_df.loc[best_model_name, "ROC_AUC"]:.4f})')

In [ ]:
# ── Visualization: Model Comparison Bar Chart ──
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('🏆 Model Performance Comparison', fontsize=15, fontweight='bold')

colors = {'Logistic Regression': '#2196F3', 'Random Forest': '#4CAF50', 'XGBoost': '#FF9800'}

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    vals  = [results[m][metric] for m in results]
    names = [m.replace(' ', '\n') for m in results]
    cols  = [colors[m] for m in results]
    bars = ax.bar(names, vals, color=cols, edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/Notebook/model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Model comparison chart saved')

In [ ]:
# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('🔍 Confusion Matrices', fontsize=15, fontweight='bold')

for i, (name, model) in enumerate(trained_models.items()):
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test.values
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Rejected','Approved'], yticklabels=['Rejected','Approved'],
                linewidths=1)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('/content/Notebook/confusion_matrices.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Confusion matrices saved')

In [ ]:
# ── ROC Curves ──
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0,1],[0,1],'k--', lw=1.2, label='Random Classifier')

for name, model in trained_models.items():
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test.values
    y_prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2.5, label=f'{name} (AUC = {auc:.3f})',
            color=list(colors.values())[list(colors.keys()).index(name)])

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('📈 ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/Notebook/roc_curves.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ ROC curves saved')

In [ ]:
# ── Feature Importance (Best Tree-Based Model) ──
tree_models = {k:v for k,v in trained_models.items() if k != 'Logistic Regression'}
best_tree   = max(tree_models, key=lambda m: results[m]['ROC_AUC'])

fi = pd.Series(
    trained_models[best_tree].feature_importances_,
    index=FEATURE_NAMES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(fi.index, fi.values, color='steelblue', edgecolor='white')
ax.set_title(f'🔑 Feature Importance — {best_tree}', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, fi.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/Notebook/feature_importance.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Feature importance chart saved')

## 💾 Step 8: Save Best Model & Metrics

In [ ]:
# ── Select Best Model ──
print('💾 SAVING BEST MODEL')
print('=' * 60)

best_model_name = max(results, key=lambda m: results[m]['ROC_AUC'])
best_model      = trained_models[best_model_name]

print(f'🏆 Best Model Selected: {best_model_name}')
print(f'   ROC-AUC  : {results[best_model_name]["ROC_AUC"]:.4f}')
print(f'   Accuracy : {results[best_model_name]["Accuracy"]:.4f}')

# Save best model
joblib.dump(best_model, '/content/Model/best_model.pkl')
print('\n✅ best_model.pkl saved')

# Save all model metrics
model_metrics = {
    'results'         : results,
    'best_model_name' : best_model_name,
    'feature_names'   : FEATURE_NAMES,
    'python_version'  : sys.version,
    'sklearn_version' : __import__('sklearn').__version__,
}
joblib.dump(model_metrics, '/content/Model/model_metrics.pkl')
print('✅ model_metrics.pkl saved')

# Save metadata as JSON too (human-readable)
meta = {
    'best_model'    : best_model_name,
    'feature_names' : FEATURE_NAMES,
    'metrics'       : {k: {m: round(v,4) for m,v in vals.items()} for k,vals in results.items()}
}
with open('/content/Model/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ metadata.json saved')

# Save all individual models too
for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(model, f'/content/Model/{safe_name}.pkl')
    print(f'✅ {safe_name}.pkl saved')

print('\n📁 All artifacts saved to /content/Model/')
print(os.listdir('/content/Model'))

In [ ]:
# ── Download all model files ──
from google.colab import files
import zipfile

# Zip the Model folder
with zipfile.ZipFile('/content/Model_artifacts.zip', 'w') as zf:
    for fname in os.listdir('/content/Model'):
        zf.write(f'/content/Model/{fname}', fname)

files.download('/content/Model_artifacts.zip')
print('\n✅ Model artifacts ready for download!')
print('   Place the unzipped files in your local Model/ directory')

## 🧪 Step 9: Inference Test

In [ ]:
# ── Test Prediction on Sample Input ──
print('🧪 INFERENCE TEST')
print('=' * 50)

# Sample applicant
sample = {
    'Gender'          : 'Male',
    'Married'         : 'Yes',
    'Dependents'      : '0',
    'Education'       : 'Graduate',
    'Self_Employed'   : 'No',
    'ApplicantIncome' : 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount'      : 150,
    'Loan_Amount_Term': 360,
    'Credit_History'  : 1.0,
    'Property_Area'   : 'Urban'
}

print(f'Sample Applicant: {sample}\n')

def predict_loan(applicant: dict) -> dict:
    """Predict loan approval for a single applicant."""
    le = joblib.load('/content/Model/label_encoders.pkl')
    sc = joblib.load('/content/Model/scaler.pkl')
    fn = joblib.load('/content/Model/feature_names.pkl')
    bm = joblib.load('/content/Model/best_model.pkl')

    d = applicant.copy()
    total  = d['ApplicantIncome'] + d['CoapplicantIncome']
    emi    = d['LoanAmount'] / max(d['Loan_Amount_Term'], 1)

    row = {
        'Gender'               : le['Gender'].transform([d['Gender']])[0],
        'Married'              : le['Married'].transform([d['Married']])[0],
        'Dependents'           : float(str(d['Dependents']).replace('+','')),
        'Education'            : le['Education'].transform([d['Education']])[0],
        'Self_Employed'        : le['Self_Employed'].transform([d['Self_Employed']])[0],
        'Loan_Amount_Term'     : d['Loan_Amount_Term'],
        'Credit_History'       : d['Credit_History'],
        'Property_Area'        : le['Property_Area'].transform([d['Property_Area']])[0],
        'Log_ApplicantIncome'  : np.log1p(d['ApplicantIncome']),
        'Log_CoapplicantIncome': np.log1p(d['CoapplicantIncome']),
        'Log_LoanAmount'       : np.log1p(d['LoanAmount']),
        'Log_TotalIncome'      : np.log1p(total),
        'EMI'                  : emi,
        'Balance_Income'       : total - emi * 1000,
        'Debt_Income_Ratio'    : d['LoanAmount'] / max(total, 1),
    }

    X_input = pd.DataFrame([row])[fn]
    best_name = joblib.load('/content/Model/model_metrics.pkl')['best_model_name']
    if best_name == 'Logistic Regression':
        X_input = sc.transform(X_input)

    pred  = bm.predict(X_input)[0]
    prob  = bm.predict_proba(X_input)[0][1]
    return {'prediction': 'Approved' if pred == 1 else 'Rejected', 'confidence': prob}

result = predict_loan(sample)
print(f'🎯 Prediction : {result["prediction"]}')
print(f'📊 Confidence : {result["confidence"]*100:.1f}%')

## ✅ Summary

| Step | Status |
|------|--------|
| Data Loading | ✅ |
| EDA | ✅ |
| Preprocessing | ✅ |
| Feature Engineering | ✅ |
| Model Training (LR, RF, XGB) | ✅ |
| Model Evaluation (5 metrics) | ✅ |
| Best Model Saved | ✅ |
| Inference Pipeline | ✅ |

---

**Next Steps:**
1. Download `Model_artifacts.zip`
2. Extract to `LoanApprovalPrediction/Model/`
3. Run `train_model.py` locally if Python version differs
4. Launch Streamlit app: `streamlit run Streamlit_App/app.py`